# 4. Gün: Risk Ölçütleri & Geriye Dönük Test
## VaR, ES, PELVE, EVT, FZ Kaybı & Backtest
### EYS'26 — Pamukkale Üniversitesi

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
from scipy.stats import norm, genpareto, t as t_dist

import sys
sys.path.insert(0, str(Path('..').resolve()))
from risk_metrics import (
    calculate_var_es, calculate_pelve_single,
    calculate_cornish_fisher_var, calculate_evt_var_es,
    backtest_var, backtest_es_acerbi_szekely, berkowitz_pit_test,
    fissler_ziegel_loss,
)

plt.style.use('dark_background')
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})

DATA_PATH = Path('..') / 'data' / 'sample_returns.csv'
df = pd.read_csv(DATA_PATH, index_col=0, parse_dates=True)
asset_cols = [c for c in df.columns if not c.endswith('_RV') and not c.endswith('_BPV')]
print(f"Varlık sayısı: {len(asset_cols)}, Gözlem: {len(df)}")

## 1. VaR & ES: Statik Hesaplama

| Yöntem | Formül |
|--------|--------|
| Parametrik Normal | μ + σ · z_{1-α} |
| Student-t | μ + s_t · t_{1-α,ν} |
| Tarihsel Simülasyon | Q_{1-α}({L_t}) |
| Cornish-Fisher | z_CF = z + (1/6)(z²-1)S + (1/24)(z³-3z)K − (1/36)(2z³-5z)S² |

In [ ]:
ASSET = asset_cols[0]
returns = df[ASSET].dropna().values
ALPHA = 0.05

rows = []
for label, method in [('Normal', 'parametric_normal'),
                       ('Student-t', 'parametric_student_t'),
                       ('Hist.Sim.', 'historical')]:
    v, e = calculate_var_es(returns, ALPHA, method)
    rows.append({'Yöntem': label, 'VaR (%)': f'{v*100:.4f}', 'ES (%)': f'{e*100:.4f}'})

cf_v, _, _, _ = calculate_cornish_fisher_var(returns, ALPHA)
cf_e = float(np.mean(-returns) + np.std(returns, ddof=1) * (norm.pdf(norm.ppf(1-ALPHA)) / ALPHA))
rows.append({'Yöntem': 'Cornish-Fisher', 'VaR (%)': f'{cf_v*100:.4f}', 'ES (%)': f'{cf_e*100:.4f}'})

pelve = calculate_pelve_single(-returns, ALPHA)
print(f"PELVE (c) = {pelve:.4f}  [Normal ref: e ≈ 2.718, Basel FRTB: 2.5]")
print(f"\nTam örneklem VaR/ES — {ASSET} (α={ALPHA}):")
pd.set_option('display.float_format', str)
pd.DataFrame(rows)

## 2. Kayan VaR Tahminleri

250 günlük kayan pencere ile her yöntemin VaR serisi hesaplanır.

In [ ]:
WINDOW = 250
n = len(returns)
T = n - WINDOW
var_d = {m: np.empty(T) for m in ['Normal', 'Student-t', 'Hist.Sim.', 'Cornish-Fisher']}
pelve_arr = np.empty(T)

for i in range(T):
    r_win = returns[i: i + WINDOW]
    for lab, mth in [('Normal','parametric_normal'),('Student-t','parametric_student_t'),
                     ('Hist.Sim.','historical')]:
        var_d[lab][i], _ = calculate_var_es(r_win, ALPHA, mth)
    var_d['Cornish-Fisher'][i], *_ = calculate_cornish_fisher_var(r_win, ALPHA)
    pelve_arr[i] = calculate_pelve_single(-r_win, ALPHA)

idx = df.index[WINDOW: WINDOW + T]
rets_aligned = returns[WINDOW:]

COLORS = {'Normal':'#E69F00','Student-t':'#56B4E9','Hist.Sim.':'#CC79A7','Cornish-Fisher':'#009E73'}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax1.plot(idx, rets_aligned, lw=0.6, color='rgba(96,165,250,0.5)', alpha=0.5, label='Getiri')
for m, c in COLORS.items():
    ax1.plot(idx, -var_d[m], lw=1.4, color=c, ls='--', label=f'VaR — {m}')
ax1.set_title(f'Kayan VaR Tahminleri — {ASSET} (α={ALPHA}, pencere={WINDOW})')
ax1.set_ylabel('Getiri / VaR'); ax1.legend(fontsize=8)

ax2.plot(idx, pelve_arr, lw=1.6, color='#fbbf24', label='PELVE (c)')
ax2.axhline(np.e, ls='--', color='#a78bfa', lw=1, label=f'Normal ref: e≈{np.e:.3f}')
ax2.axhline(2.5, ls=':', color='#f472b6', lw=1, label='Basel FRTB: 2.5')
ax2.set_title('Kayan PELVE'); ax2.set_ylabel('c'); ax2.legend()
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout(); plt.show()

## 3. EVT — Aşım Eşiği Yöntemi (POT / GPD)

Yalnızca eşik aşımları $Y = L - u$ GPD ile modellenir:
$$\mathrm{VaR}_\alpha = u + \frac{\hat{\sigma}}{\hat{\xi}}\left[\left(\frac{n}{N_u}\alpha\right)^{-\hat{\xi}}-1\right]$$

Eşik seçimi: Ortalama Aşım Grafiği — doğrusal bölge GPD geçerliliğine işaret eder.

In [ ]:
losses = -returns
THRESH_Q = 0.90
evt = calculate_evt_var_es(losses, alpha=ALPHA, threshold_quantile=THRESH_Q)
u = evt['threshold']
xi, sigma = evt['xi'], evt['sigma']

print(f"Eşik u             : {u:.6f}")
print(f"Aşım sayısı Nᵤ     : {evt['n_exceedances']}")
print(f"Şekil ξ            : {xi:.6f}")
print(f"Ölçek σ            : {sigma:.6f}")
print(f"EVT VaR (α={ALPHA}) : {evt['var']:.6f}")
print(f"EVT ES  (α={ALPHA}) : {evt['es']:.6f}")
print(f"Normal VaR (kıyas) : {calculate_var_es(returns, ALPHA, 'parametric_normal')[0]:.6f}")

if not np.isnan(xi):
    interp = ("Ağır kuyruk (Fréchet)" if xi > 0.05 else
              "Sınırlı kuyruk (Weibull)" if xi < -0.05 else
              "Hafif kuyruk (Gumbel ≈ Normal)")
    print(f"\nKuyruk yorumu: ξ = {xi:.4f} → {interp}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Ortalama aşım grafiği
u_q = np.linspace(0.60, 0.98, 40)
u_vals = np.quantile(losses, u_q)
me_vals = [float(np.mean(losses[losses > uu] - uu)) if (losses > uu).sum() >= 5 else np.nan
           for uu in u_vals]
valid = [(u_vals[i], me_vals[i]) for i in range(len(me_vals)) if not np.isnan(me_vals[i])]
if valid:
    uv, mev = zip(*valid)
    ax1.plot(uv, mev, lw=2, marker='o', ms=4, color='#34d399')
    ax1.axvline(u, color='#f472b6', ls='--', lw=1.5, label=f'Seçili u={u:.4f}')
    ax1.set_title('Ortalama Aşım Grafiği'); ax1.set_xlabel('u'); ax1.set_ylabel('E[L-u|L>u]')
    ax1.legend()

# GPD uyumu
exc_data = evt.get('exceedances', np.array([]))
if len(exc_data) > 0:
    exc_s = np.sort(exc_data)
    n_exc = len(exc_s)
    emp = np.arange(1, n_exc+1) / n_exc
    xi_f = float(xi) if not np.isnan(xi) else 0.0
    sig_f = float(sigma) if not np.isnan(sigma) else 1e-6
    theo = genpareto.cdf(exc_s, xi_f, scale=sig_f, loc=0)
    ax2.scatter(exc_s, emp, s=20, color='#60a5fa', label='Ampirik ECDF')
    ax2.plot(exc_s, theo, lw=2, color='#f472b6', label='GPD CDF (teorik)')
    ax2.set_title(f'GPD Uyumu: ξ={xi:.4f}, σ={sigma:.4f}')
    ax2.set_xlabel('Aşım Y = L − u'); ax2.set_ylabel('P(Y ≤ y)')
    ax2.legend()
plt.tight_layout(); plt.show()

## 4. Geriye Dönük Test (Backtesting)

### Test hiyerarşisi
1. **Kupiec POF** — İhlal oranı beklenen α'ya eşit mi?
2. **Christoffersen** — İhlaller zaman içinde bağımsız mı?
3. **Acerbi-Szekely Z₁/Z₂** — ES doğru tahmin ediliyor mu?
4. **Berkowitz PIT** — Tam dağılım doğru mu?

**Basel Trafik Işığı (250g, %99):** 0-4 ihlal → 🟢 Yeşil; 5-9 → 🟡 Sarı; ≥10 → 🔴 Kırmızı

In [ ]:
N_OOS = 500
METHOD = 'parametric_normal'

# Kayan tahmin
T2 = n - WINDOW
var_arr = np.empty(T2); es_arr = np.empty(T2)
for i in range(T2):
    r_win = returns[i: i+WINDOW]
    var_arr[i], es_arr[i] = calculate_var_es(r_win, ALPHA, METHOD)

ret_oos  = returns[WINDOW:][-N_OOS:]
var_oos  = var_arr[-N_OOS:]
es_oos   = es_arr[-N_OOS:]
idx_oos  = df.index[WINDOW:][-N_OOS:]

bt   = backtest_var(ret_oos, var_oos, ALPHA)
esbt = backtest_es_acerbi_szekely(ret_oos, var_oos, es_oos, ALPHA)
berk = berkowitz_pit_test(ret_oos, var_oos)

def _pr(p): return '✓ Kabul' if (p is not None and not np.isnan(float(p)) and float(p) >= 0.05) else '✗ Red'

print(f"Toplam ihlal    : {bt['violations']}  (beklenen: {round(ALPHA*N_OOS,1)})")
print(f"İhlal oranı     : {bt['violation_rate']*100:.2f}%  (hedef: {ALPHA*100:.1f}%)")
print(f"Kupiec POF      : χ²={bt['kupiec_stat']:.3f}  p={bt['kupiec_pvalue']:.4f}  {_pr(bt['kupiec_pvalue'])}")
print(f"Christoffersen  : χ²={bt['independence_stat']:.3f}  p={bt['independence_pvalue']:.4f}  {_pr(bt['independence_pvalue'])}")
print(f"Acerbi-Szekely Z1 : stat={esbt['z1_stat']:.4f}  p={esbt['z1_pvalue']:.4f}  {_pr(esbt['z1_pvalue'])}")
print(f"Berkowitz PIT   : LB p={berk['lb_pvalue_level']:.4f}  {_pr(berk['lb_pvalue_level'])}")

In [ ]:
hits = (-ret_oos > var_oos)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(idx_oos, ret_oos, lw=0.7, color='#94a3b8', alpha=0.6, label='Getiri')
ax.plot(idx_oos, -var_oos, lw=1.6, color='#E69F00', ls='--', label=f'VaR Normal (α={ALPHA})')
ax.scatter(idx_oos[hits], ret_oos[hits], color='#D55E00', s=30, zorder=5, label=f'İhlal ({hits.sum()})')
ax.set_title(f'VaR İhlal Grafiği — {ASSET} (Normal, α={ALPHA}, son {N_OOS} gün)')
ax.set_ylabel('Getiri / VaR'); ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout(); plt.show()

## 5. Fissler-Ziegel (FZ) Kaybı ile Model Seçimi

ES tek başına **elicitable değildir** — ancak (VaR, ES) çifti birlikte elicitable'dır (Fissler-Ziegel 2016).

$$S(v,e;\ell) = (\mathbf{1}_{\ell>v}-\alpha)(-v) - \mathbf{1}_{\ell>v}\ell - \frac{1}{e}\!\left(e + \frac{\ell-v}{\alpha}\mathbf{1}_{\ell>v}\right) + \log e$$

Düşük FZ kaybı → daha iyi model.

In [ ]:
N_OOS2 = 500
train_win2 = 250

var_fcs = {}; es_fcs = {}
for label, method in [('Normal','parametric_normal'),('Student-t','parametric_student_t'),
                      ('Hist.Sim.','historical')]:
    vv = []; ee = []
    for i in range(n - N_OOS2, n):
        lo = max(0, i - train_win2)
        rw = returns[lo:i] if i > 0 else returns[:1]
        v, e = calculate_var_es(rw, ALPHA, method)
        vv.append(v); ee.append(e)
    var_fcs[label] = np.array(vv); es_fcs[label] = np.array(ee)

oos_rets = returns[-N_OOS2:]
fz_losses = {}
for m in var_fcs:
    fz_losses[m] = fissler_ziegel_loss(oos_rets, var_fcs[m], es_fcs[m], ALPHA)

best = min(fz_losses, key=fz_losses.get)
print("FZ Kaybı Karşılaştırması (düşük = iyi):")
for m, v in sorted(fz_losses.items(), key=lambda x: x[1]):
    mark = " ← EN İYİ" if m == best else ""
    print(f"  {m:20s}: {v:.6f}{mark}")

## Özet

| Ölçüt | Tutarlı? | Elicitable? | Basel gereksinimi |
|-------|----------|-------------|-------------------|
| VaR | Hayır (alt-toplayıcı değil) | Evet | Basel II/III |
| ES  | Evet | Tek başına hayır | Basel IV (FRTB) |
| (VaR, ES) çifti | — | Evet (Fissler-Ziegel) | — |

**Kaynaklar:**
- Artzner vd. (1999). Coherent measures of risk. *Math. Finance*, 9(3), 203–228.
- Kupiec, P. (1995). Techniques for verifying the accuracy of risk management models. *JD*, 3(2).
- Christoffersen, P. (1998). Evaluating interval forecasts. *Int. Econ. Rev.*, 39(4), 841–862.
- Fissler, T., & Ziegel, J. F. (2016). Higher order elicitability. *Ann. Statist.*, 44(4), 1680–1707.
- Li, C., & Wang, R. (2023). PELVE. *J. Econometrics*, 234(2), 528–548.